In [3]:
import os
import unstructured
# 使用 Hugging Face 国内镜像，避免 Read timed out（必须在 import 相关库之前设置）
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"

from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    Docx2txtLoader,
    CSVLoader,
    UnstructuredMarkdownLoader,
)
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

def load_docs(path: str):
    """按扩展名选加载器，返回 Document 列表。支持: .pdf / .txt / .docx / .csv / .md"""
    ext = os.path.splitext(path)[1].lower()
    if ext == ".pdf":
        return PyPDFLoader(path).load_and_split()
    if ext == ".txt":
        return TextLoader(path, encoding="utf-8", autodetect_encoding=True).load()
    if ext == ".docx":
        return Docx2txtLoader(path).load()
    if ext == ".md":
        return UnstructuredMarkdownLoader(path).load()
    if ext == ".csv":
        return CSVLoader(path, encoding="utf-8").load()
    raise ValueError(f"不支持的后缀: {ext}，可用: .pdf .txt .docx .md .csv")

SUPPORTED_EXT = (".pdf", ".txt", ".docx", ".md", ".csv")

def list_supported_files(folder: str, recursive: bool = True):
    """收集文件夹下所有支持格式的文件路径（默认含子文件夹）。"""
    paths = []
    folder = os.path.abspath(os.path.expanduser(folder))
    if not os.path.isdir(folder):
        return paths
    for root, _, files in os.walk(folder):
        for name in files:
            if os.path.splitext(name)[1].lower() in SUPPORTED_EXT:
                paths.append(os.path.join(root, name))
        if not recursive:
            break
    return sorted(paths)

# 1. 加载：可填「文件夹」（会加载该目录及子目录下所有 .pdf/.txt/.docx/.md/.csv），或单独文件路径
folder_paths = [
    "/Users/ludan/Documents/后端知识",
]
file_paths = [
    # "/Users/ludan/Downloads/大模型导论.pdf",
    # "/Users/ludan/Downloads/深度学习.pdf",
]
all_paths = []
for folder in folder_paths:
    all_paths.extend(list_supported_files(folder, recursive=True))
all_paths.extend(file_paths)
all_paths = list(dict.fromkeys(all_paths))  # 去重保持顺序

pages = []
for path in all_paths:
    pages.extend(load_docs(path))
print(f"共加载 {len(pages)} 个文档片段，来自 {len(all_paths)} 个文件")

# 1.5 切分为较短片段再入库，避免「整页」过长导致超出模型上下文或难以定位答案
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80, length_function=len)
chunks = splitter.split_documents(pages)
print(f"切分后共 {len(chunks)} 个片段（每段约 500 字，便于检索且不超长）")

# 2. 存入 Chroma（内存模式，不写磁盘，避免沙箱/只读导致的 "readonly database"）
# 每次运行本 cell 会重新建库；若需持久化且环境可写，再改回 persist_directory=某路径
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"),
)
print("Chroma 已就绪（内存模式）")

共加载 20 个文档片段，来自 2 个文件
切分后共 52 个片段（每段约 500 字，便于检索且不超长）


/var/folders/h3/7zc3322s3y1bwclms8c2rjwc0000gn/T/ipykernel_1079/3844561461.py:77: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"),


Chroma 已就绪（内存模式）


In [4]:
def dedupe_docs_by_page(docs):
    """按 (source, page) 去重，避免同一页出现多份（常因多次写入同一 persist_directory 导致）"""
    seen = set()
    out = []
    for doc in docs:
        meta = doc.metadata
        key = (meta.get("source"), meta.get("page"), meta.get("page_label"))
        if key in seen:
            continue
        seen.add(key)
        out.append(doc)
    return out

def format_docs(docs, sep="─" * 60, dedupe=True):
    """把 LangChain Document 列表按「来源 / 页码 + 正文」格式化打印"""
    if dedupe:
        docs = dedupe_docs_by_page(docs)
    for i, doc in enumerate(docs, 1):
        meta = doc.metadata
        source = meta.get("source", "")
        page = meta.get("page") or meta.get("page_label", "?")
        print(f"{sep}")
        print(f"[{i}] 来源: {source}  页码: {page}")
        print(f"{sep}")
        print(doc.page_content)
        print()


# # 3. 检索
# docs = vectorstore.similarity_search("关于 Transformer 的细节")
# # 格式化输出检索结果（自动按来源+页码去重）
# format_docs(docs)

In [5]:
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

LM_STUDIO_BASE = "http://127.0.0.1:1234/v1"
llm = ChatOpenAI(
    base_url=LM_STUDIO_BASE,
    api_key="lm-studio",
    temperature=0,
    model="local",
)

# 优先根据参考资料回答；只有完全无关时才说「无法得出答案」（避免因上下文过长/模型保守而总说无法得出）
# QA_PROMPT = PromptTemplate(
#     template="""请根据以下「参考资料」回答问题。若资料中有与问题相关的内容，请据此简洁回答；若完全没有相关内容，再回答「根据提供的资料无法得出答案」。

# 参考资料：
# {context}

# 问题：{question}

# 请根据上述参考资料回答：""",
#     input_variables=["context", "question"],
# )
QA_PROMPT = PromptTemplate(
    template="""请根据以下「参考资料」回答问题。首先提取参考资料中的内容，根据计算机知识做一个汇总，描述资料中陈述了哪一块计算机知识。再回答问题。

参考资料：
{context}

问题：{question}

请根据上述参考资料回答：""",
    input_variables=["context", "question"],
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
pdf_qa = RetrievalQA.from_chain_type(
    llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": QA_PROMPT},
    return_source_documents=True,
)

In [6]:

query = "我想从零开始写一个CryptoHFT，我该怎么开始？"
result = pdf_qa.invoke({"query": query})
print("Answer:", result["result"])
# 展示本次回答用到的知识库片段（证明用上了 PDF）
print("\n--- 本次检索到的参考资料（来自 PDF）---")
format_docs(result["source_documents"])

Answer: **参考资料概述**

1. **CryptoHFT 系统设计参考（第 15）**  
   - 说明 CryptoHFT 为一种高频交易平台专注于加密币市场。  
   - 列出核心模块：行情采集、策略引擎、订单执行、风险控制、性能监控。  
   - 重点强调 **订单事件**（新订单、取消订单等）与系统的实时处理。  

2. **CryptoHFT 系统设计参考之核模块设计详解（第 16）**  
   - 给出核心模块细节：行情解析、策略调度、订单生成、风险评估、性能计量。  
   - 说明 50 纳秒级延迟是目标，需实现极低‑latency 的数据流与执行路径。

---

### 从零开始写一个 CryptoHFT

| 步骤 | 关键点 | 参考依据 |
|---|---|---|
| **1. 定义系统架构**  
- 采用模块化：行情采集 → 策略引擎 → 订单引擎 → 风险控制 → 性能监控。  
- 设计接口（REST/WS/HTTP）与外部 broker。 | 第 15、第 16 |
| **2. 实现行情采集**  
- 订阅多源（如 Binance, Kraken, Solana） via HTTP/WS。  
- 解析 JSON → 内存结构，保持 50 纳秒 延迟。 | 第 15 |
| **3. 策略引擎**  
- 采用事件驱动：每行情更新触发策略计算。  
- 允许多策略（如 RSI, EMA, VWMA）并可组合。 | 第 16 |
| **4. 订单生成与执行**  
- 生成 `new_order`、`cancel_order` 等事件，按策略建议。  
- 通过 broker API 发请求，记录回应时间。 | 第 15 |
| **5. 风险控制**  
- 监测持仓比例、手续费、现金流动。  
- 触发自动退让或暂停。 | 第 16 |
| **6. 性能监控**  
- 计算实时指标：latency, throughput, 成本率, 回报率。  
- 记录日志，支持回测与可视化。 | 第 15、第 16 |
| **7. 回测/模拟**  
- 用历史行情 replay 50 纳秒级时序，验证策略与订单路径。  
- 评估收益率、波动风险。 | 第 16 |
| **8. 与 broker 集

In [9]:
# DuckDuckGo MCP 相关包均要求 Python >= 3.10，若安装失败请先升级 Python 或换用 3.10+ 内核
# 可选：原版 duckduckgo-mcp-server，或维护版 duckduckgo-mcp-server-maintained
%pip install duckduckgo-mcp-server-maintained


ERROR: Ignored the following versions that require a different python version: 0.1.3 Requires-Python >=3.10
ERROR: Could not find a version that satisfies the requirement duckduckgo-mcp-server-maintained (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for duckduckgo-mcp-server-maintained
Note: you may need to restart the kernel to use updated packages.
